# Tutorial 3 — Total contribution and SHAP

**Goal:** separate two multivariable questions: how much the complete cell-type set explains jointly, and how fitted predictions are distributed among individual features.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd() if (Path.cwd() / 'tutorial_utils.py').exists() else Path.cwd() / 'tutorials'
sys.path.insert(0, str(HERE))
from tutorial_utils import find_repo_root, load_prepared_or_example, align_and_validate
from HomoloMap.stats import SpinTest
from HomoloMap.utils import run_cumulative_models, run_explanation_analysis

ROOT = find_repo_root()
N_SPINS = 100  # use >=1,000 for scientific analysis
SEED = 42
X, Y = load_prepared_or_example(ROOT, level='subclass')
X, Y = align_and_validate(X, Y, require_complete_bn=True)
spinner = SpinTest(atlas='BN', n_spins=N_SPINS, seed=SEED)

## Joint model

The total model tests the complete predictor set against rotated outcomes. It is not obtained by summing univariate correlations or significant cells.

In [ ]:
total = run_cumulative_models(
    X, Y, spinner, mode='linear', n_spins=N_SPINS,
    FDR='fdr_bh', n_jobs=1, composition_transform='none',
)
total

## Individual contributions

SHAP partitions fitted predictions among features. It measures model dependence rather than biological causality. Install `HomoloMap[explain]` and set the flag below to run it.

In [ ]:
RUN_SHAP = False

if RUN_SHAP:
    explanations = run_explanation_analysis(
        X, Y, method='shap', mode='linear', n_jobs=1,
        random_state=SEED, composition_transform='none',
    )
    first = explanations[Y.columns[0]]
    print('Total contribution:', first['total_contribution'])
    display(first['individual_ctype_contribution'].head(10))

In [ ]:
OUTPUT = ROOT / 'tutorial_outputs'
OUTPUT.mkdir(exist_ok=True)
total.to_csv(OUTPUT / 'total_models.csv')

<!-- tutorial-visual-summary -->
### Visualize total and individual contributions
The first panel summarizes joint model performance. When SHAP is enabled, the second panel ranks individual cell-type contributions.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

numeric = total.select_dtypes('number')
score_col = next((c for c in numeric.columns if 'r2' in c.lower() or 'r_sq' in c.lower()), numeric.columns[0])
fig, ax = plt.subplots(figsize=(7, 3.5))
numeric[score_col].sort_values().plot.barh(ax=ax, color='#457b9d')
ax.set(xlabel=score_col, ylabel='Brain IDP', title='Total cell-type contribution')
sns.despine()
fig.tight_layout()

if RUN_SHAP:
    contribution = first['individual_ctype_contribution']
    if isinstance(contribution, pd.DataFrame):
        contribution = contribution.select_dtypes('number').iloc[:, 0]
    contribution = contribution.sort_values().tail(15)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    contribution.plot.barh(ax=ax, color='#e9a03b')
    ax.set(xlabel='Mean absolute SHAP contribution', ylabel='Cell type',
           title=f'Individual contributions to {Y.columns[0]}')
    sns.despine()
    fig.tight_layout()
